## Comparative analysis of results in Bologna:

### Land Surface Temperature predicted from High resolution data and open Global datasets

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
# inputs
lst_observed = '../../datos-notebooks/1.HR-data-bologna/1.0.base-data/LST/LST_clipped_celsius.tif'
lst_predicted_hr_data = '../../datos-notebooks/1.HR-data-bologna/1.4.LST-prediction/LST_predicted_RF.tif'
lst_predicted_global_data = '../../datos-notebooks/2.global-data-bologna/LST-prediction/LST_predicted_RF.tif'
lst_predicted_from_hr_global_data = '../../datos-notebooks/2.global-data-bologna/LST-prediction/LST_predicted_RF_fromHRdata.tif'

#output
out_diff = '../../datos-notebooks/4.comparative-analysis/LST_difference_bologna.tif'

In [ ]:
def quantify(o, p, nodata):
    # Mask valid pixels
    mask = np.isfinite(o) & np.isfinite(p)
    if nodata is not None:
        mask &= (o != nodata)

    o = o[mask]
    p = p[mask]

    # Metrics rounded 
    print(f"Bias: {np.mean(p-o):.3f}")
    print(f"MAE: {np.mean(np.abs(p-o)):.3f}")
    print(f"RMSE: {np.sqrt(np.mean((p-o)**2)):.3f}")
    print(f"Pearson r: {np.corrcoef(o,p)[0,1]:.3f}")

#### predicted vs observed

In [ ]:
# prediction: model trained with HR data, applied with HR data
print("---- model trained with HR data vs observed LST----")
with rasterio.open(lst_observed) as s1, rasterio.open(lst_predicted_hr_data) as s2:
    quantify(s1.read(1).astype("float32"),
             s2.read(1).astype("float32"),
             s1.nodata)
    
# prediction: model trained with HR data, applied with global datasets
print("---- model trained with HR data, applied with global datasets vs observed LST ----")
with rasterio.open(lst_observed) as s1, rasterio.open(lst_predicted_from_hr_global_data) as s2:
    quantify(s1.read(1).astype("float32"),
             s2.read(1).astype("float32"),
             s1.nodata)
    
# prediction: model trained with global datasets, applied with global datasets
print("---- model trained with global datasets vs observed LST ----")
with rasterio.open(lst_observed) as s1, rasterio.open(lst_predicted_global_data) as s2:
    quantify(s1.read(1).astype("float32"),
             s2.read(1).astype("float32"),
             s1.nodata)



#### differences between predictions trained with HR data: applied with HR data and with global datasets

In [ ]:
os.makedirs(os.path.dirname(out_diff), exist_ok=True)

with rasterio.open(lst_predicted_hr_data) as src1, rasterio.open(lst_predicted_from_hr_global_data) as src2:

    # Read bands as float 
    r1 = src1.read(1).astype("float32")
    r2 = src2.read(1).astype("float32")

    # Difference 
    diff = r1 - r2

    # Save output raster 
    profile = src1.profile
    profile.update(dtype="float32")

    with rasterio.open(out_diff, "w", **profile) as dst:
        dst.write(diff.astype("float32"), 1)
        
    print("Saved in:", out_diff)

In [ ]:
# Plot result 
with rasterio.open(lst_predicted_hr_data) as src:
    xmin, ymin, xmax, ymax = src.bounds

plt.figure()
plt.title("difference between predictions")
plt.imshow(diff, cmap='viridis')
plt.colorbar(label="difference in C (HR - Global)")
plt.show()


In [ ]:
# Quantify raster differences
with rasterio.open(out_diff) as srcd:
    diff = srcd.read(1).astype("float32")
    nodata = srcd.nodata

# Build valid mask
mask = np.isfinite(diff)
if nodata is not None:
    mask &= (diff != nodata)

d = diff[mask]

# Basic stats 
print(f"Mean diff: {np.mean(d):.3f}")
print(f"Median diff: {np.median(d):.3f}")
print(f"Std diff: {np.std(d):.3f}")

# Error-like metrics 
mae = float(np.mean(np.abs(d)))
rmse = float(np.sqrt(np.mean(d**2)))
print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")


In [ ]:
# Plot bar chart with threshold labels
thresholds = [1.0, 2.0, 5.0, 10.0]
fractions = [float(np.mean(np.abs(d) > thr)) * 100 for thr in thresholds]

plt.figure()
plt.title("differences by threshold")

bars = plt.bar(range(len(thresholds)), fractions) # Create bars

plt.xlabel("Threshold")
plt.ylabel("Pixels above (%)")
plt.xticks(range(len(thresholds)), [f">{t}" for t in thresholds])
plt.show()
